# biblia-audio-conferir.ipynb — o áudio fala o mesmo que o texto?

Transcreve a narração com Whisper e compara, palavra a palavra, contra o texto
do `web-biblia.json`. É a metade que faltava da prova.

| Já provado | Como |
|---|---|
| texto baixado == texto do projeto | `biblia-texto-baixar`, similaridade 1.0000 no Mateus 2 |
| **áudio == texto** | **este notebook** |

Sem isso, o que se sabe é que dois arquivos de texto batem entre si — nada
sobre o que a voz do David Williams realmente diz. E é o áudio que manda: o
`alinhar_versiculos()` usa o texto pra derivar o tempo de cada versículo, então
texto que não corresponde ao áudio desloca o versículo **em silêncio**.

## Não existe limiar aqui, e é de propósito

No `biblia-texto-baixar` o limiar é 0,97 porque foi **medido**: duas traduções
inglesas diferentes batem ~0,83, então abaixo de 0,97 é suspeito.

Aqui não há medição equivalente. Transcrição de fala contra texto escrito nunca
dá 1,0 — Whisper erra nome próprio, engole palavra curta, ouve "Herod" como
"heroes". Um número que eu inventasse seria chute com cara de rigor.

**Então a primeira execução É a calibração.** O que vale é a lista de
diferenças, não a nota.

## As duas diferenças que importam

| O que você vê | Significa | O que fazer |
|---|---|---|
| a palavra do texto está certa e a do Whisper é que soa parecido | erro de transcrição | **ignorar** — o texto está bom |
| o áudio diz mesmo outra coisa (frase a mais, ordem trocada, versículo pulado) | áudio e texto **divergem** | investigar antes de gerar vídeo desse capítulo |

Quase tudo cai na primeira. A segunda é rara e é exatamente o que este
notebook existe pra achar.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP                                                         ║
# ╚══════════════════════════════════════════════════════════════════╝

!pip install -q openai-whisper
print('✅ whisper')

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

import shutil, sys
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO_MODULOS = Path("/content/pipeline")

if not PASTA_MODULOS.exists():
    raise SystemExit(f"❌ Módulos não encontrados: {PASTA_MODULOS}")
if DESTINO_MODULOS.exists():
    shutil.rmtree(DESTINO_MODULOS)
shutil.copytree(PASTA_MODULOS, DESTINO_MODULOS)
if str(DESTINO_MODULOS) not in sys.path:
    sys.path.insert(0, str(DESTINO_MODULOS))
print(f"✅ {len(list(DESTINO_MODULOS.glob('*.py')))} módulos")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO — edite só esta célula                          ║
# ╚══════════════════════════════════════════════════════════════════╝

# Capítulo a conferir. O livro e o capítulo saem do próprio nome
# (biblia_livros.de_nome_projeto), então não precisa repetir.
NOME_ORACAO = "40_Matt_02"

# Modelo do Whisper. Quanto maior, menos erro de transcrição -- e menos ruído
# na lista de diferenças, que é o que você vai ler.
#   base    rápido, erra bastante nome próprio
#   small   bom equilíbrio  ← recomendado pra conferência
#   medium  melhor, bem mais lento na CPU
# Com GPU ligada (Ambiente de execução → GPU), "small" leva ~1 min num
# capítulo. Sem GPU, conte alguns minutos.
MODELO_WHISPER = "small"

IDIOMA_AUDIO = "en"     # idioma falado, pra não deixar o Whisper adivinhar

# Deixe "" pra procurar sozinho na pasta do vídeo e em assets/biblia_audio/.
ARQUIVO_AUDIO = ""

print(f"Capítulo ...... {NOME_ORACAO}")
print(f"Modelo ........ {MODELO_WHISPER}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 ACHAR O ÁUDIO E O TEXTO DE REFERÊNCIA                         ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path
if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

import biblia_livros as bl
import biblia_texto as bt

BASE = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}")
PASTA_VIDEO = BASE / "videos" / NOME_ORACAO
CAMINHO_BIBLIA = BASE / "pipeline" / "dados_lexico" / "web-biblia.json"

livro, capitulo = bl.de_nome_projeto(NOME_ORACAO)
print(f"📖 {livro.nome} {capitulo}")

# ── o áudio ─────────────────────────────────────────────────────────────
# Procura onde ele costuma estar: a pasta do vídeo (narração daquele vídeo) e
# o estoque da Bíblia inteira (biblia-audio-baixar). Diz qual usou -- conferir
# o áudio errado e achar que está tudo bem é o pior resultado possível.
if ARQUIVO_AUDIO:
    audio = Path(ARQUIVO_AUDIO)
    if not audio.is_absolute():
        audio = PASTA_VIDEO / ARQUIVO_AUDIO
    candidatos = [audio]
else:
    candidatos = []
    for pasta in (PASTA_VIDEO, BASE / "assets" / "biblia_audio"):
        for ext in (".wav", ".mp3", ".m4a"):
            candidatos.append(pasta / f"{NOME_ORACAO}_audio{ext}")
            candidatos.append(pasta / f"{NOME_ORACAO}{ext}")

audio = next((c for c in candidatos if c.exists()), None)
if audio is None:
    print("❌ Não achei o áudio. Procurei:")
    for c in candidatos:
        print(f"     {c}")
    raise SystemExit("Ponha o arquivo numa dessas pastas ou preencha ARQUIVO_AUDIO.")

print(f"🔊 {audio}")
print(f"   {audio.stat().st_size/1e6:.1f} MB")

# ── o texto de referência ───────────────────────────────────────────────
texto_referencia = bt.roteiro_do_capitulo(NOME_ORACAO, CAMINHO_BIBLIA)
print(f"📄 referência: {len(texto_referencia.split())} palavras do web-biblia.json")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎧 TRANSCREVER — a parte demorada                                ║
# ╚══════════════════════════════════════════════════════════════════╝
# Guarda o resultado num .txt ao lado do áudio: reler custa milissegundos e
# transcrever custa minutos, então mexer na comparação não paga o Whisper
# de novo.

import time
from whisper_utils import carregar_modelo_whisper, detectar_device

destino_txt = PASTA_VIDEO / f"{NOME_ORACAO}_whisper_bruto_{MODELO_WHISPER}.txt"

if destino_txt.exists():
    texto_whisper = destino_txt.read_text(encoding="utf-8")
    print(f"♻️  reusando {destino_txt.name} ({len(texto_whisper.split())} palavras)")
    print("   (apague esse arquivo pra transcrever de novo)")
else:
    print(f"⏳ transcrevendo com '{MODELO_WHISPER}' em {detectar_device()}…")
    inicio = time.time()
    modelo = carregar_modelo_whisper(MODELO_WHISPER)
    resultado = modelo.transcribe(str(audio), language=IDIOMA_AUDIO, verbose=False)
    texto_whisper = resultado["text"].strip()

    PASTA_VIDEO.mkdir(parents=True, exist_ok=True)
    destino_txt.write_text(texto_whisper, encoding="utf-8")
    print(f"✅ {len(texto_whisper.split())} palavras em {time.time()-inicio:.0f}s")
    print(f"   salvo em {destino_txt.name}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 COMPARAR — áudio contra texto                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
# O comparador apaga numeração de versículo, pontuação, aspas e acento antes
# de comparar -- sobra a sequência de PALAVRAS, que é o que o alinhamento usa.
# Então tudo que aparecer aqui é diferença de palavra de verdade.

c = bt.comparar(texto_referencia, texto_whisper, contexto=4)

print(f"texto (web-biblia) ... {c.palavras_a} palavras")
print(f"áudio (whisper) ...... {c.palavras_b} palavras")
print(f"similaridade ......... {c.similaridade:.4f}")
print()

if c.identico:
    print("✅ Nem uma palavra de diferença. Raro — e ótimo.")
else:
    # ── Ordena por TAMANHO, não pela ordem no texto ──────────────────────
    # Medido com transcrição simulada sobre o Mateus 2:
    #
    #   14 erros de nome próprio  ->  similaridade 0,9675 em 14 diferenças
    #   40 palavras faltando      ->  similaridade 0,9328 em  1 diferença
    #
    # A nota sozinha engana: o caso grave "pontua melhor" que o inofensivo em
    # número de diferenças, e pior em similaridade -- as duas leituras se
    # contradizem. O que separa os dois é a FORMA: muitas diferenças pequenas
    # é ruído de transcrição; poucas e grandes é divergência de verdade.
    # Por isso o maior trecho vem primeiro, e não o primeiro do capítulo.
    def tamanho(dif):
        _, no_texto, no_audio = dif
        return max(len(no_texto.split()), len(no_audio.split()))

    ordenadas = sorted(c.diferencas, key=tamanho, reverse=True)
    grandes = [d for d in ordenadas if tamanho(d) > 12]

    print(f"{len(c.diferencas)} trecho(s) divergem.")
    if grandes:
        if len(grandes) == 1:
            print("⚠️  1 deles é GRANDE (>12 palavras) — comece por ele.")
        else:
            print(f"⚠️  {len(grandes)} deles são GRANDES (>12 palavras) — comece por esses.")
    else:
        print("   Nenhum passa de 12 palavras: cara de ruído de transcrição.")
    print()
    print("  replace = trocado   delete = falta no áudio   insert = sobra no áudio")
    print("=" * 70)

    for i, (tipo, no_texto, no_audio) in enumerate(ordenadas[:20], start=1):
        n = tamanho((tipo, no_texto, no_audio))
        marca = "  ⚠️ GRANDE" if n > 12 else ""
        print(f"\n[{i}] {tipo} · {n} palavras{marca}")
        print(f"   texto: …{no_texto}…")
        print(f"   áudio: …{no_audio}…")
    if len(ordenadas) > 20:
        print(f"\n… e mais {len(ordenadas) - 20} trecho(s) menores.")

print()
print("=" * 70)
print("COMO LER ISSO")
print("=" * 70)
print("Não há limiar aqui, de propósito: transcrição de fala contra texto")
print("escrito nunca dá 1,0, e eu não tenho medição pra dizer quanto é 'bom'.")
print("Um número inventado seria chute com cara de rigor. Esta execução é a")
print("calibração — anote e compare com a do próximo capítulo.")
print()
print("O que decide é a natureza de cada trecho:")
print("  • palavra parecida no som (Herod/heroes, Bethlehem/Beth lehem)")
print("      → erro do Whisper. O texto está certo. Ignore.")
print("  • trecho GRANDE faltando, sobrando ou fora de ordem")
print("      → áudio e texto divergem DE VERDADE. Investigue antes de fazer")
print("        vídeo deste capítulo: o alinhamento desloca em silêncio.")
print()
print("Muita diferença pequena? Suba MODELO_WHISPER pra 'medium' e apague o")
print(".txt bruto — menos ruído deixa o que importa visível.")